In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
import pickle
import time
from google.colab import files

# Mount Google Drive (optional)
from google.colab import drive
drive.mount('/content/drive')

# Upload your dataset
# Uncomment the next line if you're uploading directly to Colab instead of using Drive
# uploaded = files.upload()  # This will prompt you to upload your CSV file

# Load the dataset
# If using uploaded file:
# df = pd.read_csv('nyc_property_data.csv')
# If using Google Drive:
# df = pd.read_csv('/content/drive/MyDrive/your_path/nyc_property_data.csv')

# Let's assume we're loading the data:
print("Loading dataset...")
# Replace with your actual file path
df = pd.read_csv('nyc_property_data.csv')  

print(f"Dataset loaded with {df.shape[0]} rows and {df.shape[1]} columns")

# Display basic info
print("\nBasic dataset info:")
print(df.info())
print("\nSummary statistics:")
print(df.describe())

# Feature Engineering
print("\nPerforming feature engineering...")
start_time = time.time()

# Create zombie property indicators (based on your existing code)
df['violation_count'] = df['house_maint_code_violate_count'].fillna(0)
df['has_tax_lien'] = df['is_tax_lien_sale_eligable'].apply(lambda x: 1 if x == 1.0 else 0)
df['has_violations'] = df['violation_type'].apply(lambda x: 0 if x == 'no violations' else 1)
df['is_residential'] = df['bldgclass'].apply(lambda x: 1 if x[0] in ['A', 'B', 'C', 'D'] else 0)
df['unitsres_zero'] = df['unitsres'].apply(lambda x: 1 if x == 0.0 else 0)

# Create a preliminary zombie score
df['zombie_score'] = (df['has_violations'] * 2) +                     (df['has_tax_lien'] * 3) +                     (df['violation_count'] * 0.5) +                     (df['unitsres_zero'] * 1)

# Label high probability zombies
df['likely_zombie'] = df['zombie_score'].apply(lambda x: 1 if x > 5 else 0)

# Create borough feature for later analysis
borough_map = {1: 'Manhattan', 2: 'Bronx', 3: 'Brooklyn', 
               4: 'Queens', 5: 'Staten Island'}
df['borough'] = df['borocode'].map(borough_map)

print(f"Feature engineering completed in {time.time() - start_time:.2f} seconds")

# Analyze class distribution
zombie_distribution = df['likely_zombie'].value_counts(normalize=True) * 100
print("\nClass distribution:")
print(f"Non-zombie properties: {zombie_distribution[0]:.2f}%")
print(f"Likely zombie properties: {zombie_distribution[1]:.2f}%")

# Distribution by borough
borough_stats = df.groupby('borough')['likely_zombie'].agg(['count', 'sum'])
borough_stats['percentage'] = borough_stats['sum'] / borough_stats['count'] * 100
print("\nZombie property percentage by borough:")
print(borough_stats.sort_values('percentage', ascending=False))

# Visualize important relationships
plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
sns.countplot(x='borough', hue='likely_zombie', data=df)
plt.title('Zombie Properties by Borough')
plt.xticks(rotation=45)

plt.subplot(1, 2, 2)
sns.boxplot(x='likely_zombie', y='violation_count', data=df[df['violation_count'] < 50])
plt.title('Violation Count by Property Type')
plt.tight_layout()
plt.savefig('zombie_property_analysis.png')
plt.show()

# Prepare for model training
print("\nPreparing for model training...")
features = ['violation_count', 'has_tax_lien', 'has_violations', 
           'is_residential', 'unitsres_zero', 'borocode']

X = df[features]
y = df['likely_zombie']

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training data shape: {X_train.shape}")
print(f"Testing data shape: {X_test.shape}")

# Define models to evaluate
models = {
    "Random Forest": RandomForestClassifier(random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42)
}

# Train and evaluate models
for name, model in models.items():
    print(f"\nTraining {name}...")
    start_time = time.time()
    model.fit(X_train, y_train)
    training_time = time.time() - start_time
    
    print(f"Making predictions...")
    y_pred = model.predict(X_test)
    
    print(f"Model: {name}")
    print(f"Training time: {training_time:.2f} seconds")
    print("Classification Report:")
    print(classification_report(y_test, y_pred))
    
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))
    
    # ROC curve
    y_proba = model.predict_proba(X_test)[:,1]
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    roc_auc = auc(fpr, tpr)
    
    plt.figure(figsize=(8, 6))
    plt.plot(fpr, tpr, label=f'ROC curve (area = {roc_auc:.3f})')
    plt.plot([0, 1], [0, 1], 'k--')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'ROC Curve - {name}')
    plt.legend(loc='lower right')
    plt.savefig(f'roc_curve_{name.lower().replace(" ", "_")}.png')
    plt.show()
    
    # Feature importance
    if hasattr(model, 'feature_importances_'):
        importances = model.feature_importances_
        indices = np.argsort(importances)[::-1]
        
        plt.figure(figsize=(10, 6))
        plt.title(f'Feature Importance - {name}')
        plt.bar(range(X.shape[1]), importances[indices], align='center')
        plt.xticks(range(X.shape[1]), [features[i] for i in indices], rotation=90)
        plt.tight_layout()
        plt.savefig(f'feature_importance_{name.lower().replace(" ", "_")}.png')
        plt.show()
        
        print("Feature importance:")
        for i in indices:
            print(f"{features[i]}: {importances[i]:.4f}")

# Hyperparameter tuning for best model (Random Forest based on your results)
print("\nPerforming hyperparameter tuning for Random Forest...")
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [None, 20],
    'min_samples_split': [2, 5]
}

rf = RandomForestClassifier(random_state=42)
grid_search = GridSearchCV(estimator=rf, param_grid=param_grid, 
                          cv=3, n_jobs=-1, verbose=2, scoring='f1')

start_time = time.time()
grid_search.fit(X_train, y_train)
tuning_time = time.time() - start_time

print(f"Hyperparameter tuning completed in {tuning_time:.2f} seconds")
print(f"Best parameters: {grid_search.best_params_}")
print(f"Best F1 score: {grid_search.best_score_:.4f}")

# Train final model with best parameters
best_rf = grid_search.best_estimator_
best_rf.fit(X_train, y_train)

# Final evaluation
y_pred_final = best_rf.predict(X_test)
print("\nFinal model evaluation:")
print(classification_report(y_test, y_pred_final))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_final))

# Create probability calibration plot
plt.figure(figsize=(8, 8))
y_proba_final = best_rf.predict_proba(X_test)[:,1]

# Calculate reliability curve
from sklearn.calibration import calibration_curve
fraction_of_positives, mean_predicted_value = calibration_curve(y_test, y_proba_final, n_bins=10)

plt.plot(mean_predicted_value, fraction_of_positives, "s-", label="Random Forest")
plt.plot([0, 1], [0, 1], "k--", label="Perfectly calibrated")
plt.xlabel("Mean predicted probability")
plt.ylabel("Fraction of positives")
plt.title("Calibration plot")
plt.legend(loc="lower right")
plt.savefig("calibration_plot.png")
plt.show()

# Save the final model
print("\nSaving the final model...")
with open('zombie_detector_model.pkl', 'wb') as f:
    pickle.dump(best_rf, f)

# Download the model file
files.download('zombie_detector_model.pkl')

print("Training complete. Model saved and ready for download.")

# Also save a simple version of the data for the Streamlit app
sample_data = df.sample(1000, random_state=42)
sample_data.to_csv('sample_property_data.csv', index=False)
files.download('sample_property_data.csv')

# Create a model metadata file with feature names and thresholds
model_metadata = {
    'features': features,
    'threshold': 0.5,
    'version': '1.0',
    'trained_date': pd.Timestamp.now().strftime('%Y-%m-%d'),
    'performance': {
        'accuracy': classification_report(y_test, y_pred_final, output_dict=True)['accuracy'],
        'f1_score': classification_report(y_test, y_pred_final, output_dict=True)['1']['f1-score'],
    }
}

import json
with open('model_metadata.json', 'w') as f:
    json.dump(model_metadata, f)
files.download('model_metadata.json')

print("Model training, evaluation, and export complete!")